[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/47_paged_attention_solution.ipynb)

# 🔴 Solution: Paged Attention

vLLM-style paged KV cache — block table maps logical blocks to physical pages.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{D}}\right) V$$

where $K$ and $V$ are gathered from non-contiguous physical pages via a block table.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def paged_attention(Q, k_pages, v_pages, block_table, context_len, block_size):
    B, _, D = Q.shape
    outputs = []
    for b in range(B):
        phys_blocks = block_table[b]
        K_gathered = k_pages[phys_blocks].reshape(-1, D)
        V_gathered = v_pages[phys_blocks].reshape(-1, D)
        K_ctx = K_gathered[:context_len].unsqueeze(0)
        V_ctx = V_gathered[:context_len].unsqueeze(0)
        scale = D ** -0.5
        scores = (Q[b:b+1] @ K_ctx.transpose(-2, -1)) * scale
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V_ctx
        outputs.append(out)
    return torch.cat(outputs, dim=0)

In [ ]:
# Verify: paged output matches standard attention with contiguous layout
torch.manual_seed(0)
B, S, D = 2, 8, 16
block_size = 4
num_blocks = S // block_size

# Build contiguous K, V
K_full = torch.randn(B, S, D)
V_full = torch.randn(B, S, D)
Q = torch.randn(B, 1, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K_full.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V_full

# Build paged structures: identity block table (pages in order)
total_pages = B * num_blocks
k_pages = torch.zeros(total_pages, block_size, D)
v_pages = torch.zeros(total_pages, block_size, D)
block_table = []
for b in range(B):
    page_ids = []
    for blk in range(num_blocks):
        pid = b * num_blocks + blk
        k_pages[pid] = K_full[b, blk*block_size:(blk+1)*block_size]
        v_pages[pid] = V_full[b, blk*block_size:(blk+1)*block_size]
        page_ids.append(pid)
    block_table.append(page_ids)

paged_out = paged_attention(Q, k_pages, v_pages, block_table, context_len=S, block_size=block_size)

print('Shape:', paged_out.shape)
print('Max diff vs reference:', (paged_out - ref_out).abs().max().item())
print('Match:', torch.allclose(paged_out, ref_out, atol=1e-5))

In [ ]:
from torch_judge import check
check('paged_attention')